# Preprocessing CHIRPS V3 district rainfall dataset


Merges the per-period CHIRPS V3 district rainfall extracts into one daily
rainfall file, and resolves the duplicated MBALE records left by the zonal-stats
extraction.

**Input:** `dataset/CHIRPS_Elgon/chirps_daily_districts_*.csv` — six files,
`date, district, rain_mean, rain_max, rain_min`, catchment statistics in mm/day.

**Output:** `dataset/chirps_daily_rainfall.csv` — one row per district-day.

| Step | Effect |
| --- | --- |
| 1. Load the six extracts | 102,270 rows |
| 2. Drop Mbale Municipality | 102,270 → 92,043 |
| 3. Verify and write | 92,043 rows, 9 districts |

## 1. Load the extracts

`dataset_path` is resolved against the repository root rather than the kernel's
working directory, which depends on where Jupyter was launched.

In [ ]:
from pathlib import Path

import pandas as pd

CHIRPS_DIR = "dataset/CHIRPS_Elgon"
OUTPUT_PATH = "dataset/chirps_daily_rainfall.csv"

RAIN_COLUMNS = ["rain_mean", "rain_max", "rain_min"]


def find_repo_root(marker: str = "dataset") -> Path:
    """Walk up from the working directory until `marker` is found."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()

files = sorted((REPO_ROOT / CHIRPS_DIR).glob("chirps_daily_districts_*.csv"))
if not files:
    raise FileNotFoundError(f"No CHIRPS extracts found in {REPO_ROOT / CHIRPS_DIR}")

frames = []
for path in files:
    frame = pd.read_csv(path)
    if frame.empty:
        print(f"  skipped {path.name} (header only)")
        continue
    frame["date"] = pd.to_datetime(frame["date"])
    frame["district"] = frame["district"].str.strip().str.upper()
    # Row order within a file matters for identifying the MBALE polygons, so it
    # is preserved explicitly rather than left to depend on later sorting
    frame["source_file"] = path.name
    frame["file_row"] = range(len(frame))
    frames.append(frame)
    print(f"  {path.name:<42} {len(frame):>6} rows  "
          f"{frame['date'].min():%Y-%m-%d}..{frame['date'].max():%Y-%m-%d}")

rainfall = pd.concat(frames, ignore_index=True)

print(f"\nloaded {len(rainfall):,} rows from {len(frames)} files")
print(f"date range: {rainfall['date'].min():%Y-%m-%d} to {rainfall['date'].max():%Y-%m-%d}")
print(f"districts ({rainfall['district'].nunique()}): {', '.join(sorted(rainfall['district'].unique()))}")
print(f"nulls in rainfall columns: {int(rainfall[RAIN_COLUMNS].isna().sum().sum())}")
print(f"negative rainfall values: {int((rainfall[RAIN_COLUMNS] < 0).sum().sum())}")

### Findings — coverage

**The first file is misnamed.** `chirps_daily_districts_1996_2000.csv` contains
data from **1998-01-01**, not 1996 — there are no 1996 or 1997 records anywhere in
the extract. The combined record is therefore **1998-01-01 to 2025-12-31**
(10,227 days), not 1996–2025. Worth knowing before quoting a study period: two
years of the intended range were never extracted.

Otherwise the data is clean: no nulls, no negative values, and all nine districts
present in every file.

## 2. The duplicated MBALE records

Every date appears **twice** for MBALE and once for every other district. This is
an extraction artefact: the Uganda boundary layer used for zonal statistics
contains two features named MBALE — **Mbale Municipality** and the general
**Mbale District** — and the municipality lies *inside* the district.

The district's zonal mean therefore already includes the municipality's grid
cells. Keeping both would count the urban core twice, and averaging them would
produce a figure that is neither the district mean nor the municipality mean, so
**the municipality rows are dropped** and the district rows kept.

### Identifying which is which

The two series are not labelled, so which row is the district has to be
established from the data. Nested zonal statistics give a strict guarantee: if
the municipality's cells are a subset of the district's, then

```
district rain_max >= municipality rain_max    AND
district rain_min <= municipality rain_min
```

on **every** date — the larger polygon must bracket the smaller one's range. So
the district is identifiable as the row whose `[rain_min, rain_max]` interval
contains the other's.

Testing this against the data confirms the nesting and settles the identification.
It does not, however, resolve every date on its own: on a minority of days the two
polygons share the same min and max while their means differ, because the same
extreme values can appear in both while the cell mixes differ. The rule below
therefore uses the provable dates to determine **which row position** is the
district, then applies that position consistently — rather than deciding date by
date and leaving the ambiguous ones to a coin flip.

In [ ]:
def find_duplicated_districts(rainfall: pd.DataFrame) -> list[str]:
    """Districts appearing more than once on the same date."""
    counts = rainfall.groupby(["date", "district"]).size()
    return sorted(counts[counts > 1].index.get_level_values("district").unique())


def brackets(outer: pd.DataFrame, inner: pd.DataFrame) -> pd.Series:
    """True where `outer`'s rainfall range contains `inner`'s."""
    return (outer["rain_max"] >= inner["rain_max"]) & (outer["rain_min"] <= inner["rain_min"])


duplicated = find_duplicated_districts(rainfall)
print(f"districts with duplicate dates: {duplicated}")

# Pair the two series by their position within each date, preserving file order
mbale = rainfall[rainfall["district"] == "MBALE"].sort_values(["date", "source_file", "file_row"]).copy()
mbale["position"] = mbale.groupby("date").cumcount()
if set(mbale["position"].unique()) != {0, 1}:
    raise ValueError(f"expected exactly 2 MBALE rows per date, saw {sorted(mbale['position'].unique())}")

first = mbale[mbale["position"] == 0].set_index("date")
second = mbale[mbale["position"] == 1].set_index("date")

first_brackets = brackets(first, second)
second_brackets = brackets(second, first)
# A tie (identical min and max) satisfies both directions and cannot discriminate
tied = first_brackets & second_brackets
decisive = len(first) - int(tied.sum())

print(f"\n{len(first):,} dates with two MBALE rows")
print(f"  position 0 brackets position 1: {int(first_brackets.sum()):>6} ({100 * first_brackets.mean():.1f}%)")
print(f"  position 1 brackets position 0: {int(second_brackets.sum()):>6} ({100 * second_brackets.mean():.1f}%)")
print(f"  ties, identical min and max:    {int(tied.sum()):>6}")
print(f"  decisive dates:                 {decisive:>6}")

# On the decisive dates, one position must win every single time for the nesting
# claim to hold and for a positional rule to be safe
wins = {
    0: int((first_brackets & ~tied).sum()),
    1: int((second_brackets & ~tied).sum()),
}
print(f"\ndecisive dates won by position 0: {wins[0]:,}  by position 1: {wins[1]:,}")

district_position = max(wins, key=wins.get)
if wins[district_position] != decisive:
    raise ValueError(
        f"positional rule is unsafe: position {district_position} brackets on only "
        f"{wins[district_position]} of {decisive} decisive dates"
    )

print(f"\nposition {district_position} brackets the other on ALL {decisive:,} decisive dates")
print(f"-> position {district_position} is the general Mbale District; "
      f"position {1 - district_position} is Mbale Municipality")

In [ ]:
# Spread and extremes of the two series, as a sanity check against the other
# districts: a small urban polygon covers few grid cells and so varies less
summary = []
for position, label in [(district_position, "Mbale District"),
                        (1 - district_position, "Mbale Municipality")]:
    series = mbale[mbale["position"] == position]
    summary.append({
        "series": label,
        "mean_mm": series["rain_mean"].mean(),
        "mean_spread_mm": (series["rain_max"] - series["rain_min"]).mean(),
        "highest_max_mm": series["rain_max"].max(),
    })
for name, series in rainfall[rainfall["district"] != "MBALE"].groupby("district"):
    summary.append({
        "series": name.title(),
        "mean_mm": series["rain_mean"].mean(),
        "mean_spread_mm": (series["rain_max"] - series["rain_min"]).mean(),
        "highest_max_mm": series["rain_max"].max(),
    })

print("within-polygon spread, Mbale series against the other districts:")
print(pd.DataFrame(summary).round(2).to_string(index=False))

### Findings — MBALE identification

**The nesting is confirmed exactly.** Across 10,227 dates, position 0 brackets
position 1 on **all 9,417 decisive dates** — 100%, with no exceptions. The
remaining 810 dates are ties where both polygons report the same minimum and
maximum, which satisfies the test in both directions and so cannot discriminate;
658 of those are days with no rain anywhere in the district.

That unanimity is what makes the positional rule safe, and the cell above raises
an error rather than guessing if a future re-extraction breaks it.

The spread check corroborates it independently:

| Series | Mean rainfall | Mean spread | Highest max |
| --- | --- | --- | --- |
| **Mbale District** | 4.31 mm | **7.93 mm** | 91.4 mm |
| Mbale Municipality | 4.26 mm | **2.89 mm** | 79.4 mm |
| Other districts | 3.9–5.7 mm | 6.56–10.99 mm | — |

The district's within-polygon spread (7.93 mm) sits squarely inside the range of
its neighbours (6.56–10.99 mm), while the municipality's (2.89 mm) is less than
half the lowest of them — exactly what a small, few-celled urban polygon should
look like, and a far starker separation than the means alone suggest.

**Note on pairing.** These two series are only separable if the rows are paired in
their original file order. Sorting the MBALE rows by date alone is not enough:
pandas' default sort is not stable, so equal dates get reordered and the two
series are shuffled together. Doing that drops the bracketing test from 100% to
79% and blurs the spreads towards each other (6.8 against 4.0). The cell above
therefore sorts by `["date", "source_file", "file_row"]`, and the assertion on
the bracketing test is what catches the mistake if it recurs.

The two series correlate at 0.952 but differ by **1.08 mm on an average day**, so
this is not a cosmetic choice: picking the wrong series, or averaging them, would
shift daily rainfall by around a millimetre in the district that carries the most
flood records.

## 3. Merge, restrict to the study area, and write

Two rows are removed here:

- **Mbale Municipality**, as established above.
- **BUKWO and KWEEN.** Both were dropped from the study area during label
  preprocessing — they sit high on the Mount Elgon massif where the wet-season
  hazard is mass movement rather than riverine flooding, and between them they
  contribute 1–2 flood district-days. Retaining their rainfall would only supply
  districts that can never carry a positive.

No values are combined, so `rain_mean`, `rain_max` and `rain_min` keep their
original meaning — an average of two maxima would not have been a maximum of
anything.

In [ ]:
# Districts outside the study area, per the label preprocessing decision
EXCLUDED_DISTRICTS = ["BUKWO", "KWEEN"]

municipality_rows = mbale[mbale["position"] != district_position]

before = len(rainfall)
rainfall = rainfall.drop(index=municipality_rows.index)
outside = rainfall["district"].isin(EXCLUDED_DISTRICTS)
print(f"dropped {int(outside.sum()):,} rows for {', '.join(EXCLUDED_DISTRICTS)}")
rainfall = rainfall[~outside]
rainfall = (
    rainfall.drop(columns=["source_file", "file_row"])
    .sort_values(["district", "date"])
    .reset_index(drop=True)
)

print(f"dropped {len(municipality_rows):,} Mbale Municipality rows: {before:,} -> {len(rainfall):,}")

# The merged file must be a complete, gap-free daily panel with no duplicates
expected_days = pd.date_range(rainfall["date"].min(), rainfall["date"].max(), freq="D")
per_district = rainfall.groupby("district")["date"].agg(["count", "nunique"])
missing = {
    name: len(set(expected_days) - set(group["date"]))
    for name, group in rainfall.groupby("district")
}

print(f"\nrows: {len(rainfall):,}  districts: {rainfall['district'].nunique()}  "
      f"days per district expected: {len(expected_days):,}")
print(f"duplicate district-days: {int(rainfall.duplicated(['date', 'district']).sum())}")
incomplete = {name: gaps for name, gaps in missing.items() if gaps}
print(f"districts with missing days: {incomplete or 'none'}")
print(f"remaining duplicated districts: {find_duplicated_districts(rainfall) or 'none'}")
print()
print(per_district.to_string())

In [ ]:
OUT = REPO_ROOT / OUTPUT_PATH
rainfall.to_csv(OUT, index=False)

print(f"wrote {len(rainfall):,} rows to {OUT}")
print(f"  columns: {', '.join(rainfall.columns)}")
print(f"  {rainfall['date'].min():%Y-%m-%d} to {rainfall['date'].max():%Y-%m-%d}")
print()
print(rainfall.head(5).to_string(index=False))

## 4. Flood occurrence labels and negative filtering

Joins the flood labels onto the rainfall panel and removes the negative days on
which a flood was not physically plausible, using the threshold derived in
`empirical_rainfall_threshold.ipynb`: **5.3 mm of 15-day antecedent rainfall.**

Four decisions, all of which change the output:

1. **`Flood occurrences` is true on every day of an event's span**, not only the
   onset — the target is "was district D flooded on day T". This uses all 137
   labelled event-days rather than the 93 onsets.
2. **The panel is truncated at 2018-06-20**, where the labels end. Rainfall runs
   to 2025 but days after the labels stop are *unobserved*, not flood-free;
   marking them false would assert that no flood occurred in 2019–2025, which is
   untrue — EM-DAT records events in 2019, 2022 and 2025.
3. **The first 14 days of the record are dropped** (98 rows). A 15-day antecedent
   sum is undefined there, so those days cannot be tested against the threshold.
4. **Month-precise event-days are dropped, not labelled false.** Eight events have
   an onset dated the 1st of a month, which the threshold analysis found to be
   over-represented well beyond chance (p = 0.0069) and therefore likely a
   month-only date defaulted to the 1st. The flood probably happened that month,
   so the specific day is unknown rather than flood-free. Labelling it either way
   would inject noise. Remove the `suspect` filter below to keep them.

**Positives are never filtered.** The threshold applies to negatives only, so a
flood day below 5.3 mm is retained.

### Two further reductions, neither of them filtering

The label file holds 137 event-days but only 129 reach the panel. Both causes are
structural rather than choices, and the cell below reconciles them at runtime:

- **2 events predate the rainfall record.** `1991-09-02` MBALE and `1994-05-04`
  KAPCHORWA fall before CHIRPS begins on 1998-01-01, so there is no rainfall to
  join them to. Re-extracting CHIRPS back to 1991 — the product supports it —
  would recover both.
- **6 district-days are claimed by two events at once.** Two overlapping MBALE
  records, Serial 521 (30 days) and Serial 529 (8 days), both cover
  2010-03-04 to 2010-03-09. That is 12 event-days describing 6 district-days, and
  they collapse to 6 positives because a district-day is either flooded or it is
  not. No information is lost — the duplication was in the event list, not the
  outcome.

### Choices made here, not inherited from the brief

Decisions 1 and 2 above were specified. These were not, and each is reversible:

| Choice | Why | To undo |
| --- | --- | --- |
| Month-precise event-days dropped (8 rows) rather than labelled | An event dated the 1st probably happened *some day* that month, so labelling that specific day either way is a guess. Dropping treats it as unobserved. | Remove `~suspect` from the filter |
| Opening 14 days dropped (98 rows) | A 15-day antecedent sum is undefined there, so those days cannot be tested against the threshold at all. | Use `min_periods=1` on the rolling sum |
| `ante_15d` retained as a column | Makes the filter auditable after the fact, and is a ready-made model feature. | Drop the column before writing |
| Output named `chirps_flood_training_data.csv` | Distinguishes the model-ready frame from the consolidated rainfall record, which is still written separately. | Change `TRAINING_PATH` |

In [ ]:
FLOOD_LABELS = "dataset/flood_labelled_data.csv"
TRAINING_PATH = "dataset/chirps_flood_training_data.csv"

# From empirical_rainfall_threshold.ipynb: 15-day window, 5.3 mm, chosen to
# retain 100% of positives while removing physically implausible negatives
ANTECEDENT_WINDOW = 15
RAINFALL_THRESHOLD_MM = 5.3

labels = pd.read_csv(REPO_ROOT / FLOOD_LABELS, parse_dates=["Observation Date"])
labels["District"] = labels["District"].str.strip().str.upper()

unexpected = sorted(set(labels["District"]) - set(rainfall["district"]))
if unexpected:
    raise ValueError(f"labels reference districts absent from the rainfall panel: {unexpected}")

# An event whose ONSET is dated the 1st is month-precise, so every day of its
# span inherits that uncertainty
onsets = labels[labels["Day Index"] == 1]
month_precise_events = set(onsets.loc[onsets["Observation Date"].dt.day == 1, "Serial"])
suspect_days = set(
    zip(*labels.loc[labels["Serial"].isin(month_precise_events), ["Observation Date", "District"]].values.T)
)
flood_days = set(zip(labels["Observation Date"], labels["District"]))

label_end = labels["Observation Date"].max()
panel = rainfall[rainfall["date"] <= label_end].sort_values(["district", "date"]).reset_index(drop=True)

# Rolling sum within district, on a gap-free daily index; min_periods leaves the
# opening 14 days null rather than crediting a short sum
panel[f"ante_{ANTECEDENT_WINDOW}d"] = panel.groupby("district")["rain_mean"].transform(
    lambda s: s.rolling(ANTECEDENT_WINDOW, min_periods=ANTECEDENT_WINDOW).sum()
)

keys = list(zip(panel["date"], panel["district"]))
panel["Flood occurrences"] = [key in flood_days for key in keys]
suspect = pd.Series([key in suspect_days for key in keys], index=panel.index)

print(f"panel truncated at label end {label_end:%Y-%m-%d}: {len(panel):,} district-days")
print(f"  labelled flood district-days: {int(panel['Flood occurrences'].sum())} "
      f"of {len(labels)} in the label file")

# Reconcile the shortfall explicitly. Two mechanisms reduce 137 labelled
# event-days before any filtering, and neither is a defect:
#   - labels predating the rainfall record cannot be joined at all
#   - two events overlapping one district-day collapse to a single positive cell,
#     because a district-day is either flooded or it is not
rain_start = rainfall["date"].min()
before_rainfall = labels[labels["Observation Date"] < rain_start]
joinable = labels[labels["Observation Date"] >= rain_start]
collapsed = len(joinable) - joinable.groupby(["Observation Date", "District"]).ngroups

print(f"\n  reconciliation of {len(labels)} labelled event-days:")
print(f"    before the rainfall record starts ({rain_start:%Y-%m-%d}): -{len(before_rainfall)}")
for _, row in before_rainfall.iterrows():
    print(f"        {row['Observation Date']:%Y-%m-%d}  {row['District']}")
print(f"    overlapping events sharing a district-day:          -{collapsed}")
print(f"    = distinct positive district-days in the panel:      "
      f"{int(panel['Flood occurrences'].sum())}")
print(f"  undefined antecedent (opening {ANTECEDENT_WINDOW - 1} days): "
      f"{int(panel[f'ante_{ANTECEDENT_WINDOW}d'].isna().sum())}")
print(f"  month-precise district-days: {int(suspect.sum())} "
      f"from {len(month_precise_events)} events")

In [ ]:
before = len(panel)
panel = panel[panel[f"ante_{ANTECEDENT_WINDOW}d"].notna() & ~suspect].reset_index(drop=True)
print(f"{before:,} -> {len(panel):,} rows after dropping undefined and month-precise days")
print(f"  prevalence before filtering: {100 * panel['Flood occurrences'].mean():.3f}%")

negatives_before = int((~panel["Flood occurrences"]).sum())

# Positives are never filtered, hence the OR
plausible = (panel[f"ante_{ANTECEDENT_WINDOW}d"] >= RAINFALL_THRESHOLD_MM) | panel["Flood occurrences"]
training = panel[plausible].reset_index(drop=True)

positives = int(training["Flood occurrences"].sum())
negatives = len(training) - positives
kept_below = int(((training[f"ante_{ANTECEDENT_WINDOW}d"] < RAINFALL_THRESHOLD_MM)
                  & training["Flood occurrences"]).sum())

print(f"\nfiltered at {RAINFALL_THRESHOLD_MM} mm of {ANTECEDENT_WINDOW}-day antecedent rainfall")
print(f"  rows:      {len(training):,}")
print(f"  positives: {positives}  (all retained; {kept_below} sit below the threshold "
      "and are kept because positives are never filtered)")
print(f"  negatives: {negatives:,}  ({negatives_before - negatives:,} removed of {negatives_before:,}, "
      f"{100 * (negatives_before - negatives) / negatives_before:.1f}%)")
print(f"  prevalence: {100 * positives / len(training):.3f}%   class ratio {negatives // positives}:1")
print(f"\npositives per district:")
print(training[training["Flood occurrences"]]["district"].value_counts().to_string())

In [ ]:
OUT = REPO_ROOT / TRAINING_PATH
training.to_csv(OUT, index=False)

print(f"wrote {len(training):,} rows to {OUT}")
print(f"  columns: {', '.join(training.columns)}")
print(f"  {training['date'].min():%Y-%m-%d} to {training['date'].max():%Y-%m-%d}, "
      f"{training['district'].nunique()} districts")
print(f"  Flood occurrences dtype: {training['Flood occurrences'].dtype}")
print()
print(training[training["Flood occurrences"]].head(4).to_string(index=False))

### Findings — labelling and filtering

**48,605 rows: 121 positives against 48,484 negatives, a 400:1 ratio.**

| Stage | Rows | Positives |
| --- | --- | --- |
| Labelled event-days in `flood_labelled_data.csv` | — | 137 |
| … less 2 predating the rainfall record | — | 135 |
| … less 6 collapsing onto shared district-days | — | **129** |
| Panel truncated at 2018-06-20 | 52,332 | 129 |
| Undefined antecedent dropped (first 14 days) | 52,234 | 129 |
| Month-precise event-days dropped | 52,226 | 121 |
| Negatives below 5.3 mm removed | **48,605** | **121** |

So of the 16 labelled event-days that do not appear as positives: 2 are
unjoinable, 6 are duplicates collapsing, and 8 are the month-precise exclusion.
Only the last is a judgement call.

The filter removed **3,621 of 52,105 negatives (6.9%)**, lifting prevalence from
0.232% to 0.249%. That is a modest gain, exactly as the threshold analysis
predicted: rainfall is a weak discriminator here, and 400:1 remains far outside
the 30:1–100:1 band that would be comfortable to train on. **Handle the residual
imbalance with class weights or negative subsampling, not with a harder rainfall
cut** — the threshold analysis showed that tightening it costs positives without
materially improving the ratio.

**One positive sits below the threshold and is retained** — the `2014-03-11` MBALE
record at 5.3 mm, which is the value the threshold was set to. It is the binding
constraint on the whole filter.

Positives are reasonably spread across the seven districts (MBALE 26, BUTALEJA
24, BUDUDA 17, BULAMBULI 17, SIRONKO 16, MANAFWA 13, KAPCHORWA 8), so no district
is left unlearnable — which was the reason for dropping BUKWO and KWEEN.

## 5. Result

Two files are written:

**`dataset/chirps_daily_rainfall.csv`** — the consolidated rainfall record.
71,589 rows: 10,227 consecutive days across the 7 study districts, one row per
district-day, no duplicates and no gaps. Columns `date`, `district`,
`rain_mean`, `rain_max`, `rain_min` in mm/day, unchanged from the extraction.

**`dataset/chirps_flood_training_data.csv`** — the model-ready frame. 48,605
district-days from 1998-01-15 to 2018-06-20, with `ante_15d` and a boolean
`Flood occurrences`. 121 positives, 48,484 negatives.

Three things to carry forward:

1. **The record starts 1998, not 1996.** The first extract file is misnamed;
   there is no 1996 or 1997 data. With labels ending 2018-06-20 the usable span
   is **1998–2018**, about 20.5 years.
2. **Mbale is the district polygon only.** Its rainfall is the zonal mean over the
   whole district, which already includes the municipality. The municipality
   series is not preserved here and would need re-extracting.
3. **400:1 is still heavily imbalanced.** The rainfall filter did what it could;
   the remainder is a prevalence problem, not a filtering problem.